# Colab Fine-Tuning LLM (QLoRA via Unsloth)

This is a self-contained Google Colab notebook for fine-tuning the base model using the dataset prepared in `00_colab_dataset_download.ipynb`.

**Runtime Requirement:** Go to Runtime > Change runtime type > **T4 GPU**.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
# 2. Install dependencies (Unsloth Colab-safe install)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes
!pip install -q datasets sentencepiece pandas

In [ ]:
!pip install --no-deps unsloth_zoo

In [ ]:
!pip install trl bitsandbytes peft accelerate

In [ ]:
# 3. Import and check GPU
import torch
from unsloth import FastLanguageModel

assert torch.cuda.is_available(), "No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU"
print("GPU is available! CUDA version:", torch.version.cuda)

In [ ]:
# 4. Load Dataset from Google Drive
import pandas as pd
from datasets import Dataset
import os

DRIVE_DATA_DIR = "/content/drive/MyDrive/Ecommerce-LLM-Finetuning/data/processed"

train_df = pd.read_csv(os.path.join(DRIVE_DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DRIVE_DATA_DIR, "val.csv"))

print(f"Loaded {len(train_df)} training rows and {len(val_df)} validation rows.")

In [ ]:
# 5. Format Prompts
def format_prompts(df):
    texts = []
    for _, row in df.iterrows():
        instruction = row['instruction']
        response = row['response']
        # Using the standard instruction format defined for this project
        text = f"### Instruction:\nCustomer:\n{instruction}\n\n### Response:\n{response}"
        texts.append(text)
    return texts

train_df['text'] = format_prompts(train_df)
val_df['text'] = format_prompts(val_df)

train_dataset = Dataset.from_pandas(train_df[['text']])
val_dataset = Dataset.from_pandas(val_df[['text']])

print("Sample formatted prompt:\n")
print(train_dataset[0]['text'])

In [ ]:
# 6. Load Base Model and Apply LoRA
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # Using the base model specified in project config
    max_seq_length = max_seq_length,
    dtype = None, # Auto-detect for T4 (will use fp16)
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized in Unsloth
    bias = "none",    # Optimized in Unsloth
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 42,
)

In [ ]:
# 7. Configure and Run SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.05,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 50,
        save_strategy = "steps",
        save_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)

# Start training
trainer_stats = trainer.train()

In [ ]:
# 8. Save the Adapter to Google Drive
SAVE_ADAPTER_DIR = "/content/drive/MyDrive/Ecommerce-LLM-Finetuning/models/lora_adapter"
os.makedirs(SAVE_ADAPTER_DIR, exist_ok=True)

model.save_pretrained(SAVE_ADAPTER_DIR)
tokenizer.save_pretrained(SAVE_ADAPTER_DIR)

print(f"LoRA adapter successfully saved to {SAVE_ADAPTER_DIR}")

In [ ]:
# 9. Quick Inference Test
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

test_instruction = "Where is my order? It's been 5 days."
prompt = f"### Instruction:\nCustomer:\n{test_instruction}\n\n### Response:\n"

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
print("\n--- Model Response ---")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])